# 🏥 Hospital Bed Demand Forecasting - Data Preprocessing
This notebook handles the loading, cleaning, merging, and feature engineering for the hospital forecasting system.

In [ ]:
import pandas as pd
import numpy as np
import os

# Paths
df1 = pd.read_csv("../data/raw/hospital_insights_summary.csv")
df2 = pd.read_csv("../data/raw/healthcare_analytics_patient_flow_data.csv")
df3 = pd.read_csv("../data/raw/dataset3.csv")

## 1. Column Selection & Renaming

In [ ]:
df1 = df1[['week', 'service', 'patients_admitted', 'avg_stay', 'max_occupancy', 'staff_count', 'recommended_staff']]
df1.rename(columns={'week': 'Date', 'patients_admitted': 'Admissions', 'max_occupancy': 'Bed_Occupancy', 'staff_count': 'Staff_Count'}, inplace=True)

df2 = df2[['Patient Admission Date', 'Department Referral', 'Patient Admission Flag']]
df2.rename(columns={'Patient Admission Date': 'Date', 'Department Referral': 'Department', 'Patient Admission Flag': 'Admission_Flag'}, inplace=True)

df3 = df3[['timestamp', 'admissions', 'discharges', 'staff_count', 'flu_cases', 'bed_occupancy']]
df3.rename(columns={'timestamp': 'Date', 'admissions': 'Admissions', 'staff_count': 'Staff_Count', 'bed_occupancy': 'Bed_Occupancy', 'flu_cases': 'Flu_Cases'}, inplace=True)

## 2. Date Normalization

In [ ]:
df1['Date'] = pd.date_range(start='2019-01-01', periods=len(df1), freq='D')
df2['Date'] = pd.to_datetime(df2['Date'], dayfirst=True, errors='coerce')
df3['Date'] = pd.to_datetime(df3['Date'], errors='coerce')

for df in [df1, df2, df3]:
    df.dropna(subset=['Date'], inplace=True)
    df['Date'] = pd.to_datetime(df['Date']).dt.normalize()

## 3. Sector Mapping & Aggregation

In [ ]:
def map_sector(value):
    value = str(value).lower()
    if 'icu' in value: return 'ICU'
    elif 'emergency' in value or 'trauma' in value: return 'Emergency'
    else: return 'General'

df1['service'] = df1['service'].apply(map_sector)
df3 = df3.groupby('Date', as_index=False).agg({
    'Admissions': 'sum', 'discharges': 'sum', 'Staff_Count': 'mean', 'Flu_Cases': 'sum', 'Bed_Occupancy': 'mean'
})

## 4. Merging & Demand Generation

In [ ]:
merged_df = pd.merge(df1, df3, on='Date', how='outer', suffixes=('_df1', '_df3'))
merged_df['Admissions'] = merged_df[['Admissions_df1', 'Admissions_df3']].mean(axis=1)
merged_df['Staff_Count'] = merged_df[['Staff_Count_df1', 'Staff_Count_df3']].mean(axis=1)
merged_df['Bed_Occupancy'] = merged_df[['Bed_Occupancy_df1', 'Bed_Occupancy_df3']].mean(axis=1)
merged_df.drop(columns=['Admissions_df1', 'Admissions_df3', 'Staff_Count_df1', 'Staff_Count_df3', 'Bed_Occupancy_df1', 'Bed_Occupancy_df3'], inplace=True)
merged_df.ffill(inplace=True)

np.random.seed(42)
merged_df['ICU_Demand'] = (merged_df['Admissions'] * np.random.uniform(0.15, 0.25, len(merged_df))).round()
merged_df['Emergency_Demand'] = (merged_df['Admissions'] * np.random.uniform(0.25, 0.35, len(merged_df))).round()
merged_df['General_Demand'] = (merged_df['Admissions'] * (1 - (0.2 + 0.3))).round()

final_df = merged_df.groupby(['Date', 'service'], as_index=False).mean()
final_df.to_csv("../data/processed/final_hospital_forecasting_dataset.csv", index=False)
final_df.head()